# Sentiment Analysis with LSTM - Complete Tutorial

This notebook provides a comprehensive tutorial on using the Sentiment Analysis LSTM package.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Data Loading and Exploration](#data)
3. [Model Training](#training)
4. [Making Predictions](#predictions)
5. [Visualization](#visualization)
6. [Advanced Usage](#advanced)

## 1. Setup and Installation <a name="setup"></a>

First, let's import the necessary modules and set up our environment.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from sentiment_analysis.data_loader import DataLoader
from sentiment_analysis.model import SentimentLSTM
from sentiment_analysis.train import Trainer
from sentiment_analysis.predict import Predictor
from sentiment_analysis.visualization import Visualizer
from sentiment_analysis.config import ModelConfig

print("All imports successful!")

## 2. Data Loading and Exploration <a name="data"></a>

Let's load the IMDB dataset and explore its structure.

In [ ]:
# Initialize data loader
data_loader = DataLoader(vocab_size=10000, max_length=300)

# Load IMDB data
print("Loading IMDB dataset...")
(X_train, y_train), (X_test, y_test) = data_loader.load_imdb_data()

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Sequence length: {X_train.shape[1]}")

In [ ]:
# Get data statistics
stats = data_loader.get_data_statistics(X_train, y_train, X_test, y_test)

print("\nDataset Statistics:")
print("-" * 50)
for key, value in stats.items():
    print(f"{key:20s}: {value}")

In [ ]:
# Decode and view a sample review
sample_idx = 0
decoded_review = data_loader.decode_review(X_train[sample_idx])
sentiment = "Positive" if y_train[sample_idx] == 1 else "Negative"

print(f"\nSample Review (Index {sample_idx}):")
print(f"Sentiment: {sentiment}")
print(f"\nText: {decoded_review[:200]}...")

## 3. Model Training <a name="training"></a>

Now let's build and train our LSTM model.

In [ ]:
# Configure training parameters
ModelConfig.EPOCHS = 3  # Use fewer epochs for demo
ModelConfig.BATCH_SIZE = 128
ModelConfig.VALIDATION_SPLIT = 0.2

print("Training Configuration:")
print(f"Epochs: {ModelConfig.EPOCHS}")
print(f"Batch Size: {ModelConfig.BATCH_SIZE}")
print(f"Validation Split: {ModelConfig.VALIDATION_SPLIT}")

In [ ]:
# Initialize and train
trainer = Trainer()

print("\nStarting training...")
history = trainer.train(
    X_train,
    y_train,
    X_test,
    y_test,
    bidirectional=True,
    spatial_dropout=False,
    patience=2,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Get best epoch
best_epoch, best_val_acc = trainer.get_best_epoch()
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy: {history['test_accuracy']:.4f}")

## 4. Making Predictions <a name="predictions"></a>

Let's use our trained model to make predictions on new text.

In [ ]:
# Initialize predictor
predictor = Predictor()

# Example reviews
example_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.",
    "Terrible waste of time. The plot was predictable and the acting was mediocre at best.",
    "It was okay. Nothing special, but I didn't hate it.",
    "A masterpiece! One of the best films I've ever seen. Highly recommended!",
    "Boring and slow. I couldn't wait for it to end.",
]

print("\nPredictions on Example Reviews:")
print("=" * 80)

for i, review in enumerate(example_reviews, 1):
    result = predictor.predict_text(review)
    
    print(f"\n{i}. {review}")
    print(f"   → Sentiment: {result['sentiment']}")
    print(f"   → Score: {result['score']:.4f}")
    print(f"   → Confidence: {result['confidence']:.2%}")

## 5. Visualization <a name="visualization"></a>

Let's visualize the training history and model performance.

In [ ]:
# Create visualizer
visualizer = Visualizer(output_dir='../outputs')

# Plot training history
visualizer.plot_training_history(history)

In [ ]:
# Evaluate and get predictions for visualization
y_pred_proba = trainer.model.model.predict(X_test, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Plot prediction distribution
visualizer.plot_prediction_distribution(y_pred_proba.flatten(), y_test)

In [ ]:
# Plot confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
visualizer.plot_confusion_matrix(cm, labels=['Negative', 'Positive'])

In [ ]:
# Plot ROC curve
visualizer.plot_roc_curve(y_test, y_pred_proba.flatten())

## 6. Advanced Usage <a name="advanced"></a>

### Custom Model Architecture

In [ ]:
# Build a custom model
custom_model = SentimentLSTM()
model = custom_model.build_model(
    bidirectional=True,
    spatial_dropout=True
)

# View model summary
print(custom_model.get_model_summary())

### Interactive Prediction

In [ ]:
# Try your own review!
user_review = "Your review text here"

result = predictor.predict_text(user_review)

print(f"\nYour review: {user_review}")
print(f"Predicted sentiment: {result['sentiment']}")
print(f"Confidence: {result['confidence']:.2%}")

## Conclusion

This tutorial covered:
- Loading and exploring the IMDB dataset
- Training an LSTM model for sentiment analysis
- Making predictions on new text
- Visualizing model performance
- Advanced customization options

For more information, check out the documentation and examples in the repository!